### Load dataset

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

/Users/tsiameh/Desktop/PythonCourse/ai-ml-course/lesson20/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [2]:
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(200))

### Load Qwen3-0.6B

In [16]:
# -------------------------
# Select device
# -------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.float32)

model = model.to(device)

### Count trainable parameters

In [17]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

percentage = 100 * trainable_params / total_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {percentage:.2f}%")

Total parameters:     596,049,920
Trainable parameters: 596,049,920
Trainable percentage: 100.00%


### Check Memory used by parameters
- torch.float32
- 600 million parameters × 4 bytes ≈ 2.4 GB

In [18]:
total_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

print(f"Parameter memory: {total_bytes / 1024**3:.2f} GB")

Parameter memory: 2.22 GB


### Nice helper function

In [19]:
def print_model_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    non_trainable = total - trainable

    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Non-trainable:        {non_trainable:,}")
    print(
        f"Trainable percentage: "
        f"{100 * trainable / total:.2f}%"
    )


print_model_parameters(model)

Total parameters:     596,049,920
Trainable parameters: 596,049,920
Non-trainable:        0
Trainable percentage: 100.00%


### Tokenize the reviews

In [20]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )

In [21]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

In [22]:
print(tokenized_train[0])

{'input_ids': [40, 48859, 358, 6769, 18548, 42652, 29137, 34671, 504, 847, 2766, 3553, 1576, 315, 678, 279, 25573, 429, 22865, 432, 979, 432, 572, 1156, 5880, 304, 220, 16, 24, 21, 22, 13, 358, 1083, 6617, 429, 518, 1156, 432, 572, 30489, 553, 547, 808, 13, 34769, 421, 432, 3512, 6679, 311, 3725, 419, 3146, 11, 8916, 1660, 264, 8405, 315, 12351, 6509, 330, 772, 12563, 530, 1, 358, 2167, 1030, 311, 1490, 419, 369, 7037, 15757, 1323, 23976, 1323, 6206, 785, 7089, 374, 30188, 2163, 264, 3908, 30109, 19584, 5458, 6941, 81062, 879, 6801, 311, 3960, 4297, 1340, 646, 911, 2272, 13, 758, 3953, 1340, 6801, 311, 5244, 1059, 51209, 908, 311, 3259, 1045, 3378, 315, 24954, 389, 1128, 279, 5461, 4492, 15326, 3381, 911, 3654, 4948, 4714, 1741, 438, 279, 22500, 5004, 323, 6957, 4714, 304, 279, 3639, 4180, 13, 758, 1948, 10161, 18761, 323, 19119, 3371, 28960, 315, 52082, 911, 862, 17979, 389, 11500, 11, 1340, 702, 1839, 448, 1059, 19584, 11079, 11, 60090, 11, 323, 12224, 2953, 15757, 1323, 23976, 1323,

### Create the data collator
- for causal language modeling
- because Qwen3 is a causal language model, not a masked language model.

In [7]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

### Create the Trainer

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen3-0.6B-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    report_to="none",
    # Apple Silicon
    use_cpu=False,
    # Don't use CUDA-specific settings
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

### Start fine-tuning

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


RuntimeError: MPS backend out of memory (MPS allocated: 6.10 GB, other allocations: 699.67 MB, max allowed: 6.77 GB). Tried to allocate 6.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

### Save the model

In [ ]:
trainer.save_model("./qwen3-0.6B-imdb-final")
tokenizer.save_pretrained("./qwen3-0.6B-imdb-final")

### Test the fine-tuned model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("./qwen3-0.6B-imdb-final")

tokenizer = AutoTokenizer.from_pretrained("./qwen3-0.6B-imdb-final")

In [ ]:
prompt = "This movie was absolutely fantastic because"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=50)

In [ ]:
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

### Evaluate the model

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
# calculate perplexity

import math

perplexity = math.exp(results["eval_loss"])

print("Perplexity:", perplexity)

### Push your fine-tuned model to huggingface 

In [ ]:
from huggingface_hub import login

login()

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    push_to_hub=True,
    hub_model_id="YOUR_USERNAME/qwen3-0.6b-imdb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

trainer.push_to_hub()

### Complete code

In [ ]:
import torch
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

# -------------------------
# Select device
# -------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

# ==========================================
# 1. Configuration
# ==========================================

MODEL_NAME = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "./qwen3-imdb"
MAX_LENGTH = 512


# ==========================================
# 2. Load Dataset
# ==========================================
dataset = load_dataset("stanfordnlp/imdb")

# Small dataset for experimentation
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(200))


# ==========================================
# 3. Load Tokenizer
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# ==========================================
# 4. Load Model
# ==========================================
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model = model.to(device)

# ==========================================
# 5. Tokenize
# ==========================================
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names
)


# ==========================================
# 6. Data Collator
# ==========================================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# ==========================================
# 7. Training Arguments
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    report_to="none"
    push_to_hub=True,
    hub_model_id="YOUR_USERNAME/qwen3-0.6b-imdb"
)

# ==========================================
# 8. Trainer
# ==========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

# ==========================================
# 9. Fine-tune
# ==========================================
trainer.train()

# ==========================================
# 10. Evaluate
# ==========================================
results = trainer.evaluate()
print(results)


# ==========================================
# 11. Save
# ==========================================
trainer.save_model("./qwen3-imdb-final")

tokenizer.save_pretrained("./qwen3-imdb-final")